In [ ]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


# 01 · EDA 数据探索
> 目标：摸清数据规模、质量与业务形态，并**回收计划 §4.2 预埋的关联锚点**。
> 纪律：所有指标调用 `retailinsight`，不重算。

In [ ]:
import pandas as pd

from retailinsight.io.loader import read_items, read_parquet
from retailinsight.settings import INTERIM_DIR, ITEMS_DIR

pd.set_option("display.max_columns", 50)
members = read_parquet(INTERIM_DIR / "members.parquet")
txn = read_parquet(INTERIM_DIR / "transactions.parquet")
items = read_items(ITEMS_DIR)
products = read_parquet(INTERIM_DIR / "products.parquet")
stores = read_parquet(INTERIM_DIR / "stores.parquet")

for name, df in [("members", members), ("transactions", txn),
                 ("transaction_items", items), ("products", products), ("stores", stores)]:
    print(f"{name:20s} rows={len(df):>9,}  cols={df.shape[1]:>3}  缺失率={df.isna().mean().mean():.2%}")


## 1.1 时间覆盖与业务规模

In [ ]:
print("时间范围:", txn["txn_ts"].min(), "~", txn["txn_ts"].max())
print("会员数:", members["member_id"].nunique())
print("门店数:", stores["store_id"].nunique(), "| SKU 数:", products["sku"].nunique())
print("品类数:", products["category"].nunique())

sales = txn[(~txn["is_logic_error"]) & (~txn["is_return"])]
print("\n有效 GMV: %.1f 万元" % (sales["net_amount"].sum() / 1e4))
print("会员订单占比: %.1f%%" % (100 * (sales["member_type"] == "会员").mean()))


## 1.2 清洗前后的质量状况（质量报告同源）

In [ ]:
from retailinsight.etl.quality import missing_rates

dfs = {"members": members, "transactions": txn, "items": items}
rates = missing_rates(dfs)
print("清洗后仍存在缺失的字段：", rates if rates else "无（缺失已被转为哨兵值/标记列）")

print("\n逻辑冲突行:", int(txn["is_logic_error"].sum()))
print("退货行:", int(txn["is_return"].sum()))
print("非会员交易行:", int((txn["member_type"] != "会员").sum()))


## 1.3 埋点回收验证：4 组强关联商品组合
> 生成器预埋的 4 组组合（计划 §4.2）必须在真实挖掘结果里被找回，
> 这是「数据里确实有可被发现的故事」的证据。

In [ ]:
from retailinsight.io.schema import ANCHOR_COMBOS
from retailinsight.settings import MART_RULES

rules = read_parquet(MART_RULES)
print(f"共 {len(rules)} 条规则（lift>1.5）\n")
for a, b, _target in ANCHOR_COMBOS:
    hit = rules[(rules["antecedents"].astype(str).str.contains(a, regex=False))
                & (rules["consequents"].astype(str).str.contains(b, regex=False))]
    if len(hit):
        print(f"  ✅ {a} → {b}: lift={hit['lift'].max():.2f}  confidence={hit['confidence'].max():.2f}")
    else:
        print(f"  ❌ {a} → {b}: 未回收")


## 1.4 季节性与时段形态

In [ ]:
monthly = (sales.assign(m=sales["txn_ts"].dt.to_period("M"))
                 .groupby("m").agg(gmv=("net_amount", "sum"), orders=("txn_id", "nunique")))
monthly["aov"] = (monthly["gmv"] / monthly["orders"]).round(2)
ax = monthly["gmv"].plot(figsize=(11, 3.5), title="月度 GMV")
monthly.tail(12)
